In [2]:
import json 
import urllib.request

In [3]:
class SimpleKnowledgeGraph:
    def __init__(self):
        
        self.graph ={}
        
        
    def add_triplet(self , subject ,predicate ,obj):
        """adds a directed, labeled edge between two  entities"""
        subject = subject.strip().title()
        obj = obj.strip().title()
        predicate = predicate.strip().lower()
        print(f"Adding triples")
        if subject not in self.graph:
            print(f"Initializing the graph")
            self.graph[subject] = {}
        self.graph[subject][obj] = predicate
        
        if obj not in self.graph:
            self.graph[obj] = {}
            
    def display(self):
        """Prints the entire graph structure"""
        print('\n-- current knowledge graph---')
        for source , target in self.graph.items():
            for target , rel in target.items():
                print(f"({source}) -- [{rel}]--> ({target})")
    
    def reason_transitive_employment(self, person , company_attribute ):
        """
        Simple reasoning rule:
        if (person) --[works_at]--> (company) AND (company) -- [loaction_in] -->(location)
        Infer: (person)--[lives_in_or_near]-->(Location)
        """
        print(f"\n Reasoning for: {person}")
        if person not in self.graph:
            return "Entity not found!"
        
        for company ,rel in self.graph[person].items():
            if rel == "works_at":
                print(f"-> found explicit fact: {person} works at {company}")
                
                if company in self.graph:
                    for location , comp_rel in self.graph[company].items():
                        if comp_rel == "located_in":
                            print(f" -> Inferred Reasoned Fact: {person} is likely located  in {location}.")
        return "No new insights inferred."



In [9]:

def extract_triples_with_ollama(text, model_name="llama3.2"):
    """Uses local Ollama API to extract Subject-Predicate-Object triplets."""
    url = "http://localhost:11434/api/generate"
    prompt = f"""
    You are a strict Entity-Relationship extraction system. Extract key facts from the text as JSON triples.
    Format your response STRICTLY as a JSON array of objects with keys: "subject", "predicate", "object". reply with list of dictionary
    Do not write any introductory or concluding text. Respond ONLY with raw valid JSON.

    Text: "{text}"
    """
    data = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "format": "json" # Forces the local LLM to reply in valid JSON format
    }
    print(f"calling model for extraction")
    req = urllib.request.Request(
        url,
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res_body = json.loads(response.read().decode("utf-8"))
            response_text = res_body.get("response", "[]")
            print(f"Successfully extracted data")
            return json.loads(response_text)
    except Exception as e:
        print(f"Error communicating with local Ollama: {e}")
        return []
    

In [10]:
kg = SimpleKnowledgeGraph()
# kg.add_triplet("Alice" , "works_at","Acme Corp")
# kg.add_triplet("Acme Corp" , "located_in","San Francisco")

# kg.display()
# kg.reason_transitive_employment("Alice","located_in")        
# Example Usage to bridge extraction to construction:
raw_text = "Bob works at OpenAI. OpenAI is located in San Francisco."
extracted = extract_triples_with_ollama(raw_text)
print(f"The extracted Triplet are: {extracted}")


        

calling model for extraction
Successfully extracted data
The extracted Triplet are: {'subject': 'OpenAI', 'predicate': 'is located in', 'object': 'San Francisco'}


In [11]:
extracted


{'subject': 'OpenAI', 'predicate': 'is located in', 'object': 'San Francisco'}

In [14]:
# Pass the dictionary keys directly into the function
kg.add_triplet(extracted['subject'], extracted['predicate'], extracted['object'])

kg.display()


Adding triples
Initializing the graph

-- current knowledge graph---
(Openai) -- [is located in]--> (San Francisco)
